Python's builtin logging module solves all of the 4 standard severity levels(in increasing order of urgency):

1. DEBUG (10) : Detailed diagnostic variable dumps used only during local troubleshooting.

2. INFO(20) : Normal operational milestones(eg: "Nightly billing vatch started", "Connected to postfresql")

3. WARNING(30) : Something unexpected happened, but the app is still running (eg: "Disk usage at 85%", "Retrying slow upstream call")

4. ERROR(40) : A specific operation failed(eg: "Failed to generate PDF for Invoice #403")

5. CRITICAL (50) : Sever fialure threatening the entire appilcation (eg: "Primary database unreachable - shutting down").

### Example : COnfiguring Strucutred File + Console Logging for a Billing Batch Job

In [7]:
import logging

# Configure root logger once at application startup

logging.basicConfig(
    level = logging.INFO, # Ignores DEBUG, records INFO, WARNING, ERRROR, CRITICAL
    format = "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.FileHandler("billing_worker.log", encoding="utf-8"),
        logging.StreamHandler(), #Also prints to terminal
    ]
)

logger = logging.getLogger("billing.processor")

def process_invoke(invoice_id: str, amount: float) -> None:
    
    logger.debug("Validating payload for %s", invoice_id) #Hidden since level=INFO

    if amount <= 0:
        logger.warning("Skipping %s: non-positive amount(%.2f)", invoice_id, amount)
        return
    
    try:
        if invoice_id == "INV-99":
            raise ConnectionError("Payment gateway timed out")
        logger.info("Settled %s for INR %.2f", invoice_id, amount)
    except ConnectionError:
        # logger.exception() automatically logs at ERROR lvelel AND attaches the fulll stack trace!
        logger.exception("Failed to settle %s", invoice_id)


process_invoke("INV-101", 2400.0)
process_invoke("INV-102", -50.0)
process_invoke("INV-99", 8900.0)


2026-11-26 15:42:35 | INFO     | billing.processor | Settled INV-101 for INR 2400.00
2026-11-26 15:42:35 | WARNING  | billing.processor | Skipping INV-102: non-positive amount(-50.00)
2026-11-26 15:42:35 | ERROR    | billing.processor | Failed to settle INV-99
Traceback (most recent call last):
  File "/tmp/ipykernel_19026/2213271071.py", line 27, in process_invoke
    raise ConnectionError("Payment gateway timed out")
ConnectionError: Payment gateway timed out
